In [1]:
# ================================
# AI6130 Open-LLM Benchmark - Original vs Improved Prompt Comparison
# ================================

!pip install -q transformers datasets accelerate tqdm pandas

import datasets
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import os
import numpy as np

# -------------------------------
# Suppress auto-conversion warnings
# -------------------------------
os.environ["TRANSFORMERS_NO_AUTO_CONVERT"] = "1"

# -------------------------------
# 1️⃣ Load dataset and group by task
# -------------------------------
print("Loading Open-LLM Benchmark dataset...")
dataset = datasets.load_dataset("Open-Style/Open-LLM-Benchmark", "questions")

grouped = {}
for ex in dataset["train"]:
    d = ex["dataset"]
    if d not in grouped:
        grouped[d] = []
    grouped[d].append(ex)

print("All dataset keys detected:")
print(list(grouped.keys()))

# -------------------------------
# 2️⃣ Map dataset keys to required tasks
# -------------------------------
task_data = {}
for key in grouped.keys():
    lower_key = key.lower()
    if "commonsense" in lower_key:
        task_data["CommonsenseQA"] = grouped[key]
    if "openbook" in lower_key:
        task_data["OpenbookQA"] = grouped[key]
    if "piqa" in lower_key:
        task_data["PiQA"] = grouped[key]

print("Mapped tasks:", task_data.keys())
tasks = list(task_data.keys())
NUM_SAMPLES = 100  # limit for speed (set to None for full)

# -------------------------------
# 3️⃣ Models to evaluate
# -------------------------------
models = {
    "TinyLlama": "TinyLlama/TinyLlama_v1.1",
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "DeepSeek-1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "Qwen3-4B": "Qwen/Qwen3-4B-Instruct-2507"
}

# -------------------------------
# 4️⃣ Prompt builders
# -------------------------------
def build_prompt_original(sample):
    """Simple prompt: question + options + 'Answer:'"""
    question = sample["question"]
    options = sample["options"]
    prompt = f"{question}\n"
    for opt in options:
        prompt += f"{opt['label']}. {opt['text']}\n"
    prompt += "Answer:"
    return prompt

def build_prompt_improved(sample):
    """Few-shot with reasoning instruction (your best prompt)"""
    question = sample["question"]
    options = sample["options"]

    prompt = """
You are an expert AI system that answers multiple-choice questions.
Read the question carefully and determine the correct option.
First reason about the question briefly.
Then give the final answer using ONLY the format:
Final Answer: <LETTER>

Example 1
Question: Where do people usually sleep?
A. kitchen
B. bedroom
C. garage
D. garden
Reasoning: People usually sleep in bedrooms where beds are located.
Final Answer: B

Example 2
Question: What helps plants grow?
A. sunlight
B. plastic
C. metal
D. sand
Reasoning: Plants need sunlight for photosynthesis.
Final Answer: A

Now answer the following question.
Question:
"""

    prompt += question + "\n\nChoices:\n"

    for opt in options:
        prompt += f"{opt['label']}. {opt['text']}\n"

    prompt += """

Reasoning:
Let's think step by step.

Final Answer:
"""

    return prompt

# -------------------------------
# 5️⃣ Robust Answer Extraction
# -------------------------------
def extract_answer(text):
    """
    Extract final answer from model output.
    Returns single uppercase letter or 'X' if missing.
    """
    text = text.strip()
    if not text:
        return "X"

    if "Final Answer:" in text:
        ans = text.split("Final Answer:")[-1].strip()
    elif "Answer:" in text:
        ans = text.split("Answer:")[-1].strip()
    else:
        ans = text

    if len(ans) == 0:
        return "X"
    first_char = ans[0].upper()
    if first_char in ["A", "B", "C", "D"]:
        return first_char
    else:
        return "X"

# -------------------------------
# 6️⃣ Inference function (prompt_type determines which builder)
# -------------------------------
def infer(sample, tokenizer, model, prompt_type="improved"):
    if prompt_type == "original":
        prompt = build_prompt_original(sample)
    else:
        prompt = build_prompt_improved(sample)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,  # increased for reasoning in improved prompt
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # For original prompt, the answer is after the last "Answer:"
    # For improved, we have the structured format; extract_answer works for both.
    pred = extract_answer(full_text)
    return full_text, pred

# -------------------------------
# 7️⃣ Evaluation function (runs for one prompt type)
# -------------------------------
def evaluate(samples, tokenizer, model, prompt_type, collect_reasoning=False, max_reasoning=2):
    correct = 0
    reasoning_examples = []
    for idx, sample in enumerate(tqdm(samples, desc=f"{prompt_type}")):
        text, pred = infer(sample, tokenizer, model, prompt_type)
        if pred == sample["answerKey"].upper():
            correct += 1
        elif pred == "X":
            # Optional: print warnings
            pass

        # Collect reasoning examples only for improved prompt and only first few
        if collect_reasoning and len(reasoning_examples) < max_reasoning:
            reasoning_examples.append({
                "question": sample["question"],
                "prediction": pred,
                "answerKey": sample["answerKey"].upper(),
                "reasoning_output": text
            })
    accuracy = correct / len(samples) * 100
    return accuracy, reasoning_examples

# -------------------------------
# 8️⃣ Main benchmark loop over models and prompt types
# -------------------------------
results = {}          # results[model][task][prompt] = accuracy
reasoning_samples = {} # store a few examples for the improved prompt

for label, model_name in models.items():
    print("\n" + "="*60)
    print(f"Loading Model: {label} ({model_name})")
    print("="*60)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16
    )

    results[label] = {}
    reasoning_samples[label] = {}

    for task in tasks:
        print(f"\n--- Task: {task} ---")
        samples = task_data[task]
        if NUM_SAMPLES and NUM_SAMPLES < len(samples):
            samples = samples[:NUM_SAMPLES]

        task_results = {}
        for prompt_type in ["original", "improved"]:
            print(f"  Prompt: {prompt_type}")
            start = time.time()
            # Collect reasoning only for improved prompt
            collect = (prompt_type == "improved")
            acc, rex = evaluate(samples, tokenizer, model, prompt_type,
                                collect_reasoning=collect, max_reasoning=2)
            end = time.time()
            print(f"    Accuracy: {acc:.2f}% | Time: {end-start:.2f}s")
            task_results[prompt_type] = acc
            if collect:
                reasoning_samples[label][task] = rex

        results[label][task] = task_results

    del model
    torch.cuda.empty_cache()

# -------------------------------
# 9️⃣ Build comparison table
# -------------------------------
rows = []
for model in results:
    for task in results[model]:
        orig = results[model][task]["original"]
        impr = results[model][task]["improved"]
        diff = impr - orig
        rows.append({
            "Model": model,
            "Task": task,
            "Original (%)": round(orig, 2),
            "Improved (%)": round(impr, 2),
            "Improvement (pp)": round(diff, 2)
        })

df_results = pd.DataFrame(rows)
# Create a pivot for a cleaner view (optional)
pivot = df_results.pivot(index="Model", columns="Task", values=["Original (%)", "Improved (%)", "Improvement (pp)"])
pivot.columns = [f"{col[0]} - {col[1]}" for col in pivot.columns]
pivot = pivot.reset_index()

print("\n" + "="*80)
print("FINAL COMPARISON TABLE")
print("="*80)
print(pivot.to_string(index=False))

# Save to CSV
df_results.to_csv("benchmark_comparison.csv", index=False)
print("\nResults saved to 'benchmark_comparison.csv'")

# -------------------------------
# 10️⃣ Print sample reasoning from improved prompt
# -------------------------------
print("\n" + "="*80)
print("SAMPLE REASONING OUTPUTS (Improved Prompt)")
print("="*80)
for model in reasoning_samples:
    for task in reasoning_samples[model]:
        print(f"\n--- {model} | {task} ---")
        for ex in reasoning_samples[model][task]:
            print("Q:", ex["question"])
            print("Predicted:", ex["prediction"], "| True:", ex["answerKey"])
            # Print a snippet of the reasoning 
            snippet = ex["reasoning_output"]
            print("Model Reasoning (excerpt):\n", snippet)
            print("-"*50)

Loading Open-LLM Benchmark dataset...


README.md: 0.00B [00:00, ?B/s]

ARC.json: 0.00B [00:00, ?B/s]

CommonsenseQA.json: 0.00B [00:00, ?B/s]

Hellaswag.json: 0.00B [00:00, ?B/s]

...

    Accuracy: 80.00% | Time: 1144.73s

--- Task: PiQA ---
  Prompt: original


original: 100%|██████████| 100/100 [17:27<00:00, 10.47s/it]


    Accuracy: 52.00% | Time: 1047.23s
  Prompt: improved


improved: 100%|██████████| 100/100 [19:17<00:00, 11.57s/it]

    Accuracy: 73.00% | Time: 1157.12s

FINAL COMPARISON TABLE
        Model  Original (%) - CommonsenseQA  Original (%) - OpenbookQA  Original (%) - PiQA  Improved (%) - CommonsenseQA  Improved (%) - OpenbookQA  Improved (%) - PiQA  Improvement (pp) - CommonsenseQA  Improvement (pp) - OpenbookQA  Improvement (pp) - PiQA
DeepSeek-1.5B                          21.0                       35.0                 58.0                          22.0                       28.0                 54.0                               1.0                           -7.0                     -4.0
   Qwen2.5-3B                          36.0                       28.0                 59.0                          64.0                       67.0                 80.0                              28.0                           39.0                     21.0
     Qwen3-4B                          36.0                       39.0                 52.0                          60.0                       80.0          

In [1]:
# =============================================================================
# DeepSeek‑Specific Prompt with <think> Tags (Self‑Contained, 100 samples)
# =============================================================================

!pip install -q datasets transformers accelerate tqdm

import datasets
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------------------------
# 1. Load dataset and group tasks
# ---------------------------
print("Loading Open-LLM Benchmark dataset...")
dataset = datasets.load_dataset("Open-Style/Open-LLM-Benchmark", "questions")

grouped = {}
for ex in dataset["train"]:
    d = ex["dataset"]
    if d not in grouped:
        grouped[d] = []
    grouped[d].append(ex)

task_data = {}
for key in grouped.keys():
    lower_key = key.lower()
    if "commonsense" in lower_key:
        task_data["CommonsenseQA"] = grouped[key]
    if "openbook" in lower_key:
        task_data["OpenbookQA"] = grouped[key]
    if "piqa" in lower_key:
        task_data["PiQA"] = grouped[key]

tasks = list(task_data.keys())
print("Tasks:", tasks)

# ---------------------------
# 2. DeepSeek‑specific prompt builder
# ---------------------------
def build_prompt_deepseek(sample):
    question = sample["question"]
    options = sample["options"]
    options_text = "\n".join([f"{opt['label']}. {opt['text']}" for opt in options])

    system = "You are an expert assistant that solves multiple-choice questions by reasoning step by step."
    user = f"{question}\n\nOptions:\n{options_text}\n\nPlease reason step by step inside <think> tags, then provide your final answer as a single letter inside <answer> tags."

    # ChatML format (works for DeepSeek‑R1‑Distill)
    prompt = f"<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n<think>"
    return prompt

def extract_answer_deepseek(text):
    """Extract answer from <answer> tags, fallback to capital letter."""
    if "<answer>" in text and "</answer>" in text:
        ans = text.split("<answer>")[-1].split("</answer>")[0].strip()
    else:
        # fallback: look for "Final Answer:" or any capital letter
        if "Final Answer:" in text:
            ans = text.split("Final Answer:")[-1].strip()
        else:
            ans = text.strip()
    if len(ans) == 0:
        return "X"
    # Take first uppercase letter if present
    for ch in ans:
        if ch.isupper() and ch in "ABCD":
            return ch
    return "X"

# ---------------------------
# 3. Evaluate DeepSeek only
# ---------------------------
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"\n--- Loading {model_id} with DeepSeek‑specific prompt ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()

num_samples = 50
results = {}
examples = []

for task in tasks:
    samples = task_data[task][:num_samples]
    if not samples:
        continue
    correct = 0
    for idx, sample in enumerate(tqdm(samples, desc=f"DeepSeek {task}")):
        prompt = build_prompt_deepseek(sample)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=2000,          # allow long reasoning
                do_sample=True,               # deterministic
                pad_token_id=tokenizer.eos_token_id
            )
        input_len = inputs['input_ids'].shape[1]
        generated = outputs[0][input_len:]
        full_text = tokenizer.decode(generated, skip_special_tokens=True)
        pred = extract_answer_deepseek(full_text)
        if pred == sample["answerKey"].upper():
            correct += 1
        # collect first two examples for inspection
        if len(examples) < 2:
            examples.append({
                "task": task,
                "question": sample["question"],
                "true": sample["answerKey"],
                "pred": pred,
                "output": full_text   
            })
    acc = correct / len(samples) * 100
    results[task] = acc
    print(f"  {task}: {acc:.2f}%")

del model
torch.cuda.empty_cache()

# ---------------------------
# 4. Print results
# ---------------------------
print("\n" + "="*60)
print("DeepSeek with <think> prompt (100 samples, max_new_tokens=200)")
print("="*60)
for task in tasks:
    print(f"{task}: {results[task]:.2f}%")

print("\nExample outputs:")
for ex in examples:
    print(f"\nTask: {ex['task']}")
    print(f"Q: {ex['question']}")
    print(f"True: {ex['true']}, Predicted: {ex['pred']}")
    print("Output snippet:\n", ex['output'])

Loading Open-LLM Benchmark dataset...


README.md: 0.00B [00:00, ?B/s]

ARC.json: 0.00B [00:00, ?B/s]

CommonsenseQA.json: 0.00B [00:00, ?B/s]

Hellaswag.json: 0.00B [00:00, ?B/s]

...

  CommonsenseQA: 20.00%


DeepSeek OpenbookQA: 100%|██████████| 50/50 [20:14<00:00, 24.29s/it]


  OpenbookQA: 42.00%


DeepSeek PiQA: 100%|██████████| 50/50 [15:33<00:00, 18.67s/it]

  PiQA: 44.00%

DeepSeek with <think> prompt (100 samples, max_new_tokens=200)
CommonsenseQA: 20.00%
OpenbookQA: 42.00%
PiQA: 44.00%

Example outputs:

Task: CommonsenseQA
Q: A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?
True: A, Predicted: X
Output snippet:
 
Okay, so I'm trying to figure out this multiple-choice question about revolving doors and how they serve as a security measure at a specific location. The options are a bank, a library, a department store, a mall, and a new york. 

First, I know that a revolving door is a common security feature in many places because it's easy to lose access if someone doesn't remember the code. But here, it's also serving as a security measure, which means it's intended to prevent unauthorized access. 

I'm thinking about where these revolving doors are typically found. They're pretty standard in most commercial buildings, like retail stores, banks, libraries, and department stores.